# Gato: ¿cuánto ayuda la memoria?

Este cuaderno usa el mismo `gato.py` de la clase y del juego en línea (https://ferdiex.github.io/gato/). Vamos a medir cuánto trabajo hace minimax en el gato, cuánto se ahorra con memoria y por qué el agente nunca pierde.

Ejecuta las celdas en orden con **Mayúsculas + Enter**.

In [ ]:
# Descarga gato.py si no está en la carpeta (en Colab no está al principio)
import os, urllib.request
if not os.path.exists("gato.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ferdiex/gato/main/gato.py", "gato.py")

import time
import gato
print("gato.py cargado")

## 1. Resolver el gato completo

`gato.minimax` tiene memoria (`@lru_cache`). Le pedimos el valor del tablero vacío: con juego perfecto, ¿quién gana?

In [ ]:
gato.minimax.cache_clear()          # empezamos sin nada en memoria

inicio = time.perf_counter()
valor, jugada = gato.minimax("-" * 9)
segundos = time.perf_counter() - inicio

print(f"Valor del tablero vacío: {valor}   (0 = empate)")
print(f"Mejor primera jugada para X: casilla {jugada}")
print(f"Tiempo: {segundos:.3f} s")
print(gato.minimax.cache_info())

`misses` es el número de tableros que minimax tuvo que **calcular**: cada tablero distinto, una sola vez. `hits` son las veces que la respuesta ya estaba en memoria y se reutilizó.

## 2. El mismo cálculo sin memoria

Escribimos un minimax idéntico, pero sin `@lru_cache`, que además cuenta cuántas veces se llama. Usa las mismas reglas de `gato.py`.

In [ ]:
llamadas = 0

def minimax_sin_memoria(board):
    global llamadas
    llamadas += 1
    if gato.terminal(board):
        return gato.utility(board), None
    player = gato.to_move(board)
    pairs = []
    for i in gato.moves(board):
        child = gato.play(board, i, player)
        pairs.append((minimax_sin_memoria(child)[0], i))
    return max(pairs) if player == "x" else min(pairs)

inicio = time.perf_counter()
print(minimax_sin_memoria("-" * 9))
segundos = time.perf_counter() - inicio
print(f"Llamadas sin memoria: {llamadas:,}")
print(f"Tableros distintos (con memoria): {gato.minimax.cache_info().misses:,}")
print(f"Tiempo sin memoria: {segundos:.2f} s")

**Pregunta:** ¿cuántas veces más trabajo hace minimax sin memoria? Calcula el cociente entre las dos cifras.

## 3. ¿De dónde salen las llamadas repetidas?

Dos órdenes distintos de jugadas pueden llegar al mismo tablero. Sin memoria, minimax resuelve ese tablero una vez por cada camino que llega a él.

In [ ]:
def jugar(jugadas):
    board = "-" * 9
    for i in jugadas:
        board = gato.play(board, i, gato.to_move(board))
    return board

a = jugar([0, 1, 4])     # X en 0, O en 1, X en 4
b = jugar([4, 1, 0])     # X en 4, O en 1, X en 0
gato.show(a)
print("¿Es el mismo tablero?", a == b)

**Ejercicio:** encuentra otro par de secuencias que lleguen al mismo tablero.

## 4. El agente nunca pierde

Recorremos **todas** las formas en que una persona puede jugar contra el agente. En cada turno humano probamos todas las casillas libres; en cada turno del agente, este juega su mejor jugada.

In [ ]:
def todas_las_partidas(board, agente):
    """Devuelve (partidas, derrotas del agente)."""
    if gato.terminal(board):
        u = gato.utility(board)
        perdio = (u == 1 and agente == "o") or (u == -1 and agente == "x")
        return 1, int(perdio)
    player = gato.to_move(board)
    if player == agente:
        _, i = gato.minimax(board)
        return todas_las_partidas(gato.play(board, i, player), agente)
    total = derrotas = 0
    for i in gato.moves(board):
        t, d = todas_las_partidas(gato.play(board, i, player), agente)
        total, derrotas = total + t, derrotas + d
    return total, derrotas

for agente in ("o", "x"):
    t, d = todas_las_partidas("-" * 9, agente)
    print(f"Agente con {agente.upper()}: {t} partidas posibles, {d} derrotas")

## 5. ¿Por qué a veces no se apura a ganar?

Con utilidad de solo +1, 0 y −1, ganar ahora o ganar dentro de tres jugadas vale lo mismo. Contamos las posiciones donde el agente (O) podría ganar de inmediato y elige otra jugada.

In [ ]:
def gana_ya(board, player):
    return [i for i in gato.moves(board)
            if gato.winner(gato.play(board, i, player)) == player]

def posiciones_del_agente(board, agente, vistas):
    if gato.terminal(board):
        return
    player = gato.to_move(board)
    if player == agente:
        vistas.add(board)
        _, i = gato.minimax(board)
        posiciones_del_agente(gato.play(board, i, player), agente, vistas)
    else:
        for i in gato.moves(board):
            posiciones_del_agente(gato.play(board, i, player), agente, vistas)

vistas = set()
posiciones_del_agente("-" * 9, "o", vistas)
lentas = [b for b in vistas if gana_ya(b, "o") and gato.minimax(b)[1] not in gana_ya(b, "o")]
print(f"Posiciones donde decide el agente: {len(vistas)}")
print(f"Podía ganar ya y no lo hizo: {len(lentas)}")
gato.show(lentas[0])
print("Juega en", gato.minimax(lentas[0])[1], "pero podía ganar en", gana_ya(lentas[0], "o"))

**Ejercicio:** cambia la utilidad para premiar las victorias rápidas, por ejemplo ±(10 − fichas en el tablero), y vuelve a contar. La celda siguiente es un punto de partida.

In [ ]:
from functools import lru_cache

def utilidad_rapida(board):
    fichas = 9 - board.count("-")
    w = gato.winner(board)
    return (10 - fichas) if w == "x" else -(10 - fichas) if w == "o" else 0

@lru_cache(maxsize=None)
def minimax_rapido(board):
    if gato.terminal(board):
        return utilidad_rapida(board), None
    player = gato.to_move(board)
    pairs = [(minimax_rapido(gato.play(board, i, player))[0], i) for i in gato.moves(board)]
    return max(pairs) if player == "x" else min(pairs)

lentas2 = [b for b in vistas if gana_ya(b, "o") and minimax_rapido(b)[1] not in gana_ya(b, "o")]
print(f"Con la utilidad rápida, victorias inmediatas desperdiciadas: {len(lentas2)}")

## Para pensar

1. Sin memoria, minimax hace más de medio millón de llamadas, pero solo hay 5,478 tableros distintos. ¿Qué parte del árbol de la lámina 7 se repite?
2. En la página del juego, vacía la memoria del agente y haz que empiece él. Compara el tiempo de su primera jugada con el de las siguientes. ¿Por qué cambia tanto?
3. Alfa-beta y la memoria ahorran trabajo de formas distintas. ¿Cuál trabajo evita cada una?